# 05 - Pipeline 4: Fine-tuned Transformer

Notebook này fine-tune Transformer cho bài toán phân loại Human (`0`) và AI-generated (`1`) trên Google Colab hoặc Kaggle GPU T4.

Báo cáo EDA của dataset ghi nhận 487.235 dòng; trung vị là 390 từ cho Human và 338 từ cho AI, IQR 279–470 từ. Vì vậy giới hạn 256 subword tokens có nguy cơ cắt mất phần đáng kể của văn bản. Cấu hình mặc định dùng `MAX_LENGTH=512`, là context window tiêu chuẩn của model này; batch size được giảm để phù hợp VRAM của T4. Notebook cũng đo phân bố token thực tế bằng tokenizer và in tỷ lệ văn bản sẽ bị cắt ở 256, 384 và 512 tokens trước khi train.

- Model mặc định: `distilroberta-base`.
- Có thể đổi sang `microsoft/deberta-v3-small` để thử model mạnh hơn.
- Validation dùng để chọn threshold; test chỉ được đánh giá sau khi đã khóa threshold.

Trước khi chạy, bật GPU T4 trong Colab hoặc Kaggle. Kaggle cần bật Internet hoặc thêm model pretrained vào Input nếu model chưa được cache.

In [ ]:
# Cài các package còn thiếu. Colab/Kaggle thường đã có torch và một phần dependencies.
import importlib.util
import subprocess
import sys

required = {
    'transformers': 'transformers>=4.46,<5',
    'datasets': 'datasets>=3.0',
    'accelerate': 'accelerate>=1.0',
    'evaluate': 'evaluate>=0.4',
    'sklearn': 'scikit-learn>=1.3',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

print('Dependency check completed.')

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    Trainer, TrainingArguments, set_seed
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Transformer fine-tuning on CPU will be very slow. Enable a T4/P100 GPU first.')

## 1. Cấu hình và tìm các tập đã chia

Pipeline 4 dùng lại đúng ba file `train.csv`, `val.csv`, `test.csv` trong thư mục `data/processed`; notebook không đọc lại CSV gốc và không tự chia lại. Đặt `DATA_DIR` nếu thư mục dữ liệu của bạn ở vị trí khác.

Colab: upload cả ba file vào một thư mục, hoặc mount Drive và trỏ `DATA_DIR` tới thư mục đó. Kaggle: thêm dataset chứa thư mục `processed` bằng **Add Input**; notebook sẽ tự tìm bộ ba file.

In [ ]:
# ===== Điều chỉnh các biến này trước khi train =====
DATA_DIR = None  # Ví dụ: '/content/data/processed' hoặc '/kaggle/input/<dataset>/processed'
MODEL_NAME = 'distilroberta-base'  # Thử 'microsoft/deberta-v3-small' sau baseline
MAX_SAMPLES = None  # Giữ toàn bộ các dòng của split train đã chuẩn bị
MAX_LENGTH = 512  # T4: context dài; dynamic padding giúp tiết kiệm VRAM
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4  # Effective train batch size = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TOKEN_LENGTH_AUDIT_PER_CLASS = 2_500

IS_KAGGLE = Path('/kaggle').exists()
OUTPUT_DIR = Path('/kaggle/working/pipeline4_transformer') if IS_KAGGLE else Path('/content/pipeline4_transformer')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_data_dir(manual_dir=None):
    required_files = ['train.csv', 'val.csv', 'test.csv']
    if manual_dir:
        path = Path(manual_dir)
        if path.is_dir() and all((path / name).is_file() for name in required_files):
            return path
        raise FileNotFoundError(f'DATA_DIR phải chứa train.csv, val.csv, test.csv: {path}')

    candidates = [Path('data/processed'), Path('/content/data/processed'), Path('/content/processed')]
    if IS_KAGGLE:
        candidates.extend(path.parent for path in Path('/kaggle/input').rglob('train.csv'))
    for path in candidates:
        if path.is_dir() and all((path / name).is_file() for name in required_files):
            return path
    return None


DATA_DIR = resolve_data_dir(DATA_DIR)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Không tìm thấy đủ train.csv, val.csv, test.csv. Hãy upload/add input cả 3 file hoặc đặt DATA_DIR.'
    )

print('Processed data:', DATA_DIR)
print('Output        :', OUTPUT_DIR)

## 2. Nạp các tập train / validation / test có sẵn

Transformer nhận văn bản gốc; không áp dụng Regex cleaning của Pipeline 1. Đọc trực tiếp ba CSV đã được chia sẵn, giữ nguyên ranh giới split, không xóa trùng/lấy mẫu/chia lại trong notebook. Validation dùng để chọn checkpoint và threshold; test chỉ mở ở bước đánh giá cuối.

In [ ]:
started = time.perf_counter()

def load_split(filename):
    frame = pd.read_csv(DATA_DIR / filename, usecols=['text', 'target'])
    frame = frame.rename(columns={'target': 'label'})
    frame['text'] = frame['text'].astype('string').str.strip()
    frame['label'] = pd.to_numeric(frame['label'], errors='coerce')
    if frame['text'].isna().any() or frame['text'].str.len().le(0).any():
        raise ValueError(f'{filename} contains missing or empty text; review the prepared split.')
    if frame['label'].isna().any() or not frame['label'].isin([0, 1]).all():
        raise ValueError(f'{filename} target must contain only 0 and 1.')
    frame['label'] = frame['label'].astype('int64')
    return frame

train_df = load_split('train.csv')
validation_df = load_split('val.csv')
test_df = load_split('test.csv')

print(f'Loaded existing splits in {time.perf_counter() - started:.1f}s')
for split_name, frame in [('Train', train_df), ('Validation', validation_df), ('Test', test_df)]:
    counts = frame['label'].value_counts().sort_index().rename(index={0: 'Human', 1: 'AI'})
    print(f'{split_name}: {len(frame):,} rows | ' + ', '.join(f'{label}={count:,}' for label, count in counts.items()))

print('No resplitting or resampling: all rows from the prepared CSV files are retained.')

## 3. Đo token length và tokenize

Số từ trong báo cáo EDA là xấp xỉ, còn Transformer dùng subword tokens. Cell này đo tokenizer length trên một mẫu cân bằng 5.000 bài, in p50/p90/p95/p99 và tỷ lệ bị cắt ở các giới hạn thường dùng. Cấu hình mặc định 512 được đối chiếu với số đo đó trước khi train.

Dynamic padding chỉ pad đến bài dài nhất trong batch, giảm VRAM so với pad mọi bài lên 512.

In [ ]:
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(validation_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Đo token length trên mẫu phân tầng train trước khi áp dụng truncation.
probe_n = min(TOKEN_LENGTH_AUDIT_PER_CLASS, int(train_df['label'].value_counts().min()))
length_probe = train_df.groupby('label', group_keys=False).sample(n=probe_n, random_state=RANDOM_STATE)
probe_tokens = tokenizer(length_probe['text'].tolist(), truncation=False, padding=False)['input_ids']
probe_lengths = np.asarray([len(ids) for ids in probe_tokens])
length_summary = pd.Series(probe_lengths).describe(percentiles=[0.50, 0.90, 0.95, 0.99])
display(length_summary.rename('subword_tokens').to_frame().round(1))
for limit in [256, 384, 512]:
    print('Văn bản vượt {:>3} tokens: {:.1%}'.format(limit, np.mean(probe_lengths > limit)))

if MAX_LENGTH > tokenizer.model_max_length:
    raise ValueError(f'MAX_LENGTH={MAX_LENGTH} vượt giới hạn tokenizer ({tokenizer.model_max_length}).')

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = raw_datasets.map(tokenize, batched=True, remove_columns=['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if torch.cuda.is_available() else None)

print(tokenized_datasets)
print('Tokenizer vocabulary size:', tokenizer.vocab_size)

## 4. Fine-tune

Best checkpoint được chọn theo `macro_f1` trên validation. Cấu hình khởi đầu cho T4 là 512 tokens, batch 4, eval batch 8 và gradient accumulation 4 (effective batch 16). Nếu CUDA out-of-memory, giảm train batch còn 2 và tăng accumulation lên 8 để giữ effective batch.

In [ ]:
id2label = {0: 'Human', 1: 'AI-generated'}
label2id = {'Human': 0, 'AI-generated': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

def softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    probabilities = softmax(logits)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], zero_division=0
    )
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': float((f1[0] + f1[1]) / 2),
        'ai_precision': float(precision[1]),
        'ai_recall': float(recall[1]),
        'pr_auc': average_precision_score(labels, probabilities),
        'roc_auc': roc_auc_score(labels, probabilities),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

started = time.perf_counter()
train_result = trainer.train()
print('Training completed in {:.1f} minutes.'.format((time.perf_counter() - started) / 60))
display(pd.DataFrame([train_result.metrics]))

## 5. Chọn threshold trên validation

Threshold 0.5 không luôn tối ưu cho Macro-F1. Cell này thử các threshold 0.05–0.95 trên validation, ưu tiên Macro-F1 rồi AI Recall khi hòa điểm. Test vẫn chưa được dùng ở bước này.

In [ ]:
validation_prediction = trainer.predict(tokenized_datasets['validation'])
validation_probabilities = softmax(validation_prediction.predictions)[:, 1]
validation_labels = validation_prediction.label_ids

threshold_rows = []
for threshold in np.arange(0.05, 0.951, 0.01):
    predicted = (validation_probabilities >= threshold).astype(int)
    _, recall, f1, _ = precision_recall_fscore_support(
        validation_labels, predicted, labels=[0, 1], zero_division=0
    )
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'macro_f1': float((f1[0] + f1[1]) / 2),
        'ai_recall': float(recall[1]),
    })

threshold_table = pd.DataFrame(threshold_rows).sort_values(
    ['macro_f1', 'ai_recall'], ascending=False
).reset_index(drop=True)
BEST_THRESHOLD = float(threshold_table.loc[0, 'threshold'])
print(f'Selected validation threshold: {BEST_THRESHOLD:.2f}')
display(threshold_table.head(10))

## 6. Đánh giá cuối cùng trên test và export model

Bây giờ mới dự đoán test bằng checkpoint tốt nhất và threshold đã khóa từ validation. Export bao gồm tokenizer, weights/config, threshold và metrics để dùng lại khi deploy.

In [ ]:
test_prediction = trainer.predict(tokenized_datasets['test'])
test_probabilities = softmax(test_prediction.predictions)[:, 1]
test_labels = test_prediction.label_ids
test_predictions = (test_probabilities >= BEST_THRESHOLD).astype(int)

precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_predictions, labels=[0, 1], zero_division=0
)
test_metrics = {
    'accuracy': float(accuracy_score(test_labels, test_predictions)),
    'macro_f1': float((f1[0] + f1[1]) / 2),
    'ai_precision': float(precision[1]),
    'ai_recall': float(recall[1]),
    'ai_f1': float(f1[1]),
    'pr_auc': float(average_precision_score(test_labels, test_probabilities)),
    'roc_auc': float(roc_auc_score(test_labels, test_probabilities)),
    'threshold': BEST_THRESHOLD,
}

print('Final test metrics:')
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(test_labels, test_predictions, target_names=['Human', 'AI-generated'], digits=4))

EXPORT_DIR = OUTPUT_DIR / 'best_transformer'
trainer.save_model(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))

run_summary = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'max_samples': MAX_SAMPLES,
    'train_rows': len(train_df),
    'validation_rows': len(validation_df),
    'test_rows': len(test_df),
    'best_threshold': BEST_THRESHOLD,
    'test_metrics': test_metrics,
}
(EXPORT_DIR / 'run_summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')

print('Saved Transformer model to:', EXPORT_DIR)
print('Kaggle: file nằm ở /kaggle/working để download sau khi run. Colab: nén thư mục rồi download hoặc copy vào Google Drive.')

## 7. Dự đoán thử một văn bản

Hàm dưới đây dùng model vừa fine-tune và `BEST_THRESHOLD`. Model Transformer chỉ nên được đưa vào Extension/API sau khi so sánh công bằng với Pipeline 1–3 về Macro-F1, AI Recall, latency và kích thước model.

In [ ]:
@torch.inference_mode()
def predict_text(text):
    encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    encoded = {key: value.to(trainer.model.device) for key, value in encoded.items()}
    logits = trainer.model(**encoded).logits.detach().cpu().numpy()
    ai_probability = float(softmax(logits)[0, 1])
    prediction = int(ai_probability >= BEST_THRESHOLD)
    return {
        'label': id2label[prediction],
        'ai_probability': round(ai_probability, 4),
        'threshold': BEST_THRESHOLD,
    }

example = 'Artificial intelligence is changing how people learn, work, and communicate.'
predict_text(example)